In [1]:
!pip install sentence-transformers faiss-cpu rdflib transformers torch

In [71]:
import os
import pickle
import faiss
import numpy as np

from sentence_transformers import SentenceTransformer

from rdflib import Graph

from transformers import pipeline

In [72]:
BASE_PATH = r"D:\1 Univesrity work\Lect 2\Data semantics\Knowledge-Graph-Enhanced-RAG-System-for-Academic-Question-Answering-in-a-Data-Science-Curriculum"

VECTOR_DB_PATH = os.path.join(BASE_PATH, "vector_db")
KG_PATH = os.path.join(BASE_PATH, "knowledge_graph")
OUTPUT_PATH = os.path.join(BASE_PATH, "outputs")

In [73]:
index = faiss.read_index(
    os.path.join(VECTOR_DB_PATH, "faiss_index.index")
)

print("FAISS index loaded!")

FAISS index loaded!


In [74]:
import pickle
import os

with open(
    os.path.join(VECTOR_DB_PATH, "metadata.pkl"),
    "rb"
) as f:

    chunks = pickle.load(f)

print(f"Loaded {len(chunks)} chunks")

Loaded 4619 chunks


In [75]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded!


In [76]:
kg = Graph()

kg.parse(
    os.path.join(KG_PATH, "academic_kg.ttl"),
    format="turtle"
)

print(f"KG loaded with {len(kg)} triples")

KG loaded with 11 triples


In [9]:
# 🔥 STEP 1 — UPGRADE TRANSFORMERS
!pip install -U transformers accelerate sentencepiece

In [77]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import warnings

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
warnings.filterwarnings("ignore")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [78]:
import torch

def generate_answer(prompt):

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)

    outputs = model.generate(
        **inputs,
        max_new_tokens=200
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [79]:
def retrieve_chunks(query, top_k=5):

    query_embedding = embedding_model.encode([query])

    distances, indices = index.search(
        np.array(query_embedding).astype("float32"),
        top_k
    )

    retrieved = []

    for idx in indices[0]:

        text = chunks[idx]["chunk_text"]

        # CLEANING STEP 👇
        if len(text) > 200:
            text = text[:500]  # trim noise

        retrieved.append(text)

    return retrieved

In [80]:
print(model.config.model_type)

t5


In [81]:
inputs = tokenizer("What is RDF?", return_tensors="pt")
output = model.generate(**inputs, max_new_tokens=50)
print(tokenizer.decode(output[0], skip_special_tokens=True))

a non-profit organisation


In [82]:
def retrieve_kg_facts(query):

    query = query.lower()

    # simple normalization
    query = query.replace("what is", "")
    query = query.replace("?", "").strip()

    facts = []

    for s, p, o in kg:

        s_name = s.split("/")[-1].lower()
        p_name = p.split("/")[-1].lower()
        o_name = o.split("/")[-1].lower()

        # better matching logic
        if (
            query in s_name
            or query in o_name
            or s_name in query
            or o_name in query
        ):
            facts.append(f"{s_name} → {p_name} → {o_name}")

    return facts

In [83]:
retrieve_kg_facts("What is Neo4j?")

['graphdatabase → implementedby → neo4j']

In [84]:
def build_context(query):

    retrieved_chunks = retrieve_chunks(query)

    kg_facts = retrieve_kg_facts(query)

    context = "\n\n".join(retrieved_chunks[:3])  # only top 3
    if kg_facts:

        context += "\n\nKnowledge Graph Facts:\n"

        context += "\n".join(kg_facts)

    return context

In [85]:
def answer_question(query):

    context = build_context(query)

    prompt = f"""
    Answer the question using the context below.

    Context:
    {context}

    Question:
    {query}

    Answer:
    """

    response = generator(
        prompt,
        max_new_tokens=256
    )

    return response[0]["generated_text"]

In [104]:
print(answer_question("What is RDF?"))

NameError: name 'chunks_df' is not defined

In [87]:
print(retrieve_chunks("RDF"))

['della valle - http:applied-semantic-web.org rdf-sowl in a nutshell rdfs semantics (core part of it) if then x rdfs:subclassof y . a rdf:type y . a rdf:type x . x rdfs:subclassof y . x rdfs:subclassof z . y rdfs:subclassof z . x p y . x q y . p rdfs:subpropertyof q . p rdfs:subpropertyof q . p rdfs:subpropertyof r . q rdfs:subpropertyof r . x p y . x rdf:type z . p rdfs:domain z . x p u . u rdf:type z . p rdfs:range z . read out more in rdf semantics http:www.w3.orgtrrdf-mt 24 artificial intellig', 'intended for human consumption, but also as (rdf) structured data that machines can locate, retrieve, combine, validate, reason over, query over, etc., towards solving tasks automatically. conceptually, the web of data is then composed of graphs of data published on individual web-pages, where one can click on a node or edge-label  or more precisely perform a http lookup on an iri of the graph  to be transported to another graph elsewhere on the web with relevant content on that node or ed

In [111]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

generator = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer
)

print("LLM READY")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


KeyError: "Unknown task text2text-generation, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']"

In [112]:
import pandas as pd

df = pd.read_excel("D:\1 Univesrity work\Lect 2\Data semantics\Knowledge-Graph-Enhanced-RAG-System-for-Academic-Question-Answering-in-a-Data-Science-Curriculum/processed/chunked_data.xlsx")

print(df.shape)
print(df.columns)

OSError: [Errno 22] Invalid argument: 'D:\x01 Univesrity work\\Lect 2\\Data semantics\\Knowledge-Graph-Enhanced-RAG-System-for-Academic-Question-Answering-in-a-Data-Science-Curriculum/processed/chunked_data.xlsx'

In [95]:
def retrieve_chunks(query, top_k=5):

    query_embedding = embedding_model.encode([query])

    distances, indices = index.search(
        query_embedding.astype("float32"),
        top_k
    )

    results = []

    for idx in indices[0]:
        results.append(df.iloc[idx]["chunk_text"])

    return results

In [96]:
def build_context(query):

    chunks = retrieve_chunks(query)
    kg_facts = retrieve_kg_facts(query)

    context = "\n\n".join(chunks[:3])

    if kg_facts:
        context += "\n\nKG:\n" + "\n".join(kg_facts)

    return context

In [97]:
def answer_question(query):

    context = build_context(query)

    prompt = f"""
Answer the question clearly and simply.

Context:
{context}

Question:
{query}

Answer:
"""

    result = generator(
        prompt,
        max_new_tokens=200
    )

    return result[0]["generated_text"]

In [98]:
print(answer_question("What is RDF?"))

NameError: name 'chunks_df' is not defined